> **ARCHIVE — superseded.** This notebook is kept as a record of the research path, not as working code. Credentials were removed and outputs cleared. Results in it may rest on defects documented in `docs/RESEARCH_LOG.md` (stage: v1 audit, survivor pool, no costs). Comments are partly in Vietnamese. The maintained implementation is `src/memescan/`.

# So sanh cac phien ban chon universe

Cung mot bo luat entry/exit, **chi khac cach chon ro co phieu**. Muc dich: tach ra xem
gia tri den tu dau — tu luat giao dich, hay tu viec chon ten.

**Khong look-ahead:** moi feature `.shift(1)`, FINRA tru 12 ngay cong bo, Reddit/options tru 1 ngay,
vao lenh tai `open[i+1]`. Xem Cell 2 in ra do phu song truoc khi tin bat ky ket qua nao.

> `compare_versions.py` la ban chay headless cua dung notebook nay (dung khi muon chay qua dem).

In [ ]:
# Cell 1 - Setup + load du lieu
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

C = Path("meme_cache"); ALLOC, CAP, MAXPOS = 10000.0, 100000.0, 10
def load(p):
    p = Path(p)
    if p.exists():
        try: return pd.read_parquet(p)
        except Exception: pass
    pk = Path(str(p)+".pkl")
    return pd.read_pickle(pk) if pk.exists() else None

pool = load(C/"pool_universe.parquet").sort_values(["ticker","date"])
memeset = set(load(C/"bt_daily.parquet")["ticker"].unique())
cl = pool.pivot(index="date", columns="ticker", values="close")
vo = pool.pivot(index="date", columns="ticker", values="vol")
hi = pool.pivot(index="date", columns="ticker", values="high")
lo = pool.pivot(index="date", columns="ticker", values="low")
op = pool.pivot(index="date", columns="ticker", values="open")
dv = cl*vo; DATES = list(cl.index); TICKS = list(cl.columns)
zs = lambda d: d.sub(d.mean(axis=1), axis=0).div(d.std(axis=1).replace(0, np.nan), axis=0)

def align_daily(df_wide, target):
    """Khop theo NGAY LICH. Pool co timestamp 05:00 con mentions/options co 00:00 ->
       reindex truc tiep se ra 0 ngay giao nhau va feature bi zero AM THAM."""
    d = df_wide.copy(); d.index = pd.DatetimeIndex(d.index).normalize()
    d = d[~d.index.duplicated(keep="last")]
    o = d.reindex(pd.DatetimeIndex(target).normalize()); o.index = target
    return o

print(f"Pool: {len(TICKS)} ma | {len(DATES)} phien | {DATES[0].date()} -> {DATES[-1].date()}")
print(f"  trong do {len(memeset & set(TICKS))} ma tu seed meme (hindsight), con lai ngau nhien")

In [ ]:
# Cell 2 - Dung feature + BAO CAO DO PHU SONG (doc ky truoc khi tin ket qua)
dv_s, cl_s, hi_s, lo_s = dv.shift(1), cl.shift(1), hi.shift(1), lo.shift(1)
tr_ = pd.concat([hi_s-lo_s, (hi_s-cl_s.shift(1)).abs(), (lo_s-cl_s.shift(1)).abs()]).groupby(level=0).max()
BASE = (zs(dv_s.rolling(5).mean()/dv_s.rolling(60).mean()).fillna(0)
        + zs((tr_.rolling(5).mean()/cl_s)/(tr_.rolling(60).mean()/cl_s)).fillna(0)
        + .5*zs(cl_s/cl_s.shift(20)-1).fillna(0)
        + .5*zs(-np.log(cl_s.clip(lower=.01))).fillna(0))
LIQ = dv_s.rolling(20).mean() >= 1e6

# ---- SHORT INTEREST (FINRA, tru 12 ngay cong bo) ----
SI_CHG = pd.DataFrame(0.0, index=DATES, columns=TICKS); SI_DTC = SI_CHG.copy(); HAS_SI = False
si = load(C/"finra_si_history.parquet")
if si is not None and len(si):
    si = si.sort_values(["ticker","settle"]).copy()
    si["pub"] = si["settle"] + pd.Timedelta(days=12)
    si["chg"] = si.groupby("ticker")["short_int"].pct_change()
    dnp = np.array(DATES, dtype="datetime64[ns]")
    for tk, g in si.groupby("ticker"):
        if tk not in TICKS: continue
        g = g.dropna(subset=["pub"]).sort_values("pub")
        idx = np.searchsorted(g["pub"].values.astype("datetime64[ns]"), dnp, side="right")-1
        ok = idx >= 0
        if ok.any():
            SI_CHG.loc[ok, tk] = np.nan_to_num(g["chg"].values[idx[ok]])
            if "dtc" in g: SI_DTC.loc[ok, tk] = np.nan_to_num(g["dtc"].values[idx[ok]])
    HAS_SI = True
    print(f"SHORT INTEREST : {si.ticker.nunique()} ma | {si.settle.min().date()} -> {si.settle.max().date()}")
else: print("SHORT INTEREST : khong co")

# ---- SOCIAL (Reddit) + lam sach extractor ----
SOC_Z = pd.DataFrame(0.0, index=DATES, columns=TICKS); SOC_AC = SOC_Z.copy(); HAS_SOC = False
rm = load(C/"reddit_mentions.parquet")
if rm is not None and len(rm):
    rm = rm[rm["ticker"].isin(TICKS)]
    M = align_daily(rm.pivot_table(index="date", columns="ticker", values="mentions", aggfunc="sum"),
                    DATES).reindex(columns=TICKS)
    cover = M.notna().any(axis=1); M = M.fillna(0.0)
    if cover.sum() == 0:
        print("SOCIAL         : !!! 0 ngay khop - lech timestamp, BO QUA")
    else:
        # extractor cu .upper() ca cau -> tu tieng Anh thanh ticker. Loc bang do BUNG NO.
        try: words = {w.strip().upper() for w in open("/usr/share/dict/words") if 2 <= len(w.strip()) <= 5}
        except Exception: words = set()
        Mc = M.loc[cover]
        burst = {t: (Mc[t].max()/max(Mc[t][Mc[t]>0].median(),1)) if (Mc[t]>0).any() else 0 for t in TICKS}
        BLOCK = [t for t in TICKS if t in words and burst[t] < 10 and (Mc[t]>0).mean() > .6]
        PROXY = [t for t in ("BE","ANY","HE") if t in M.columns]
        proxy = M[PROXY].sum(axis=1) if PROXY else pd.Series(0.0, index=M.index)
        M[BLOCK] = 0.0; nfix = 0
        if proxy.std() > 0:
            for t in TICKS:
                if t in words and t not in BLOCK:
                    b = np.polyfit(proxy, M[t], 1)[0]
                    if b > 0: M[t] = (M[t]-b*proxy).clip(lower=0); nfix += 1
        Ms = np.log1p(M).shift(1)
        # IC do duoc: accel20 = +0.025 (DUONG) | level = -0.064 (AM -> da dong nguoi = vao muon)
        SOC_AC = zs(Ms - Ms.rolling(20, min_periods=5).mean()).fillna(0)
        SOC_Z  = zs(Ms.rolling(20, min_periods=3).mean()).fillna(0)
        HAS_SOC = True
        print(f"SOCIAL         : {rm.ticker.nunique()} ma | {int(cover.sum())}/{len(DATES)} phien "
              f"({cover.sum()/len(DATES):.0%}) | chan {len(BLOCK)} ma nhiem, khu nhiem {nfix} ma")
else: print("SOCIAL         : chua co du lieu")

# ---- OPTIONS FLOW ----
OPT = pd.DataFrame(0.0, index=DATES, columns=TICKS); HAS_OPT = False
of = load(C/"options_flow.parquet")
if of is not None and len(of):
    of = of[of["ticker"].isin(TICKS)]
    O = align_daily(of.pivot_table(index="date", columns="ticker", values="otm_call_vol", aggfunc="sum"),
                    DATES).reindex(columns=TICKS)
    covo = O.notna().any(axis=1); O = O.fillna(0.0)
    if covo.sum() == 0: print("OPTIONS        : !!! 0 ngay khop - BO QUA")
    else:
        Os = np.log1p(O).shift(1)
        # IC do duoc: OTM call LEVEL = +0.061 (manh nhat) | accel20 = +0.005 (vo dung)
        # -> dung LEVEL. Khac social: options mua deu = dealer hedge = ap luc mua co hoc.
        OPT = zs(Os.rolling(20, min_periods=5).mean()).fillna(0); HAS_OPT = True
        print(f"OPTIONS        : {of.ticker.nunique()} ma | {int(covo.sum())}/{len(DATES)} phien "
              f"({covo.sum()/len(DATES):.0%}, chi tu 2024)")
else: print("OPTIONS        : chua co du lieu")

# ---------- E. PAST-MEME (hop le: chi dot da KET THUC truoc t) ----------
# Bias cu = chon ten nam 2026 roi backtest tu 2021. Bien the HOP LE: tai moi ngay t chi
# dem cac dot meme da co DINH truoc t (dot bat dau ngay d chi biet duoc o ngay dinh d+<=20).
# IC vo dieu kien = -0.15 (past-meme roi rung) NHUNG co dieu kien khi ignition bat -> tot hon.
PAST_N = pd.DataFrame(0.0, index=DATES, columns=TICKS)
for sym in TICKS:
    c = cl[sym].values; n = len(c); t = 0; peaks = []
    while t < n - 1:
        if np.isnan(c[t]): t += 1; continue
        w = cl[sym].values[t+1:min(t+21, n)]; w = w[~np.isnan(w)]
        if len(w) == 0: t += 1; continue
        if w.max()/c[t] - 1 >= 0.50:
            seg = c[t+1:min(t+21, n)]; pk = t + 1 + int(np.nanargmax(seg))
            peaks.append(pk); t = pk + 1
        else: t += 1
    if peaks:
        pk = np.array(peaks); col = TICKS.index(sym)
        for i in range(n):
            k = (pk < i).sum()                      # CHI dot da co dinh TRUOC i
            if k: PAST_N.iloc[i, col] = k
PASTM = zs(PAST_N).fillna(0).shift(1)
print(f"PAST-MEME      : {int((PAST_N>0).any().sum())} ma tung co dot meme")



In [ ]:
# Cell 3 - INFORMATION COEFFICIENT: do tin hieu trUOC khi backtest
# IC dung hang chuc NGHIN quan sat, dang tin hon nhieu so voi backtest ~200 lenh.
def ic_table(feats, mask=None, horizons=(5,10,20)):
    idx = DATES if mask is None else list(pd.Index(DATES)[mask])
    rows = []
    for name, F in feats.items():
        r = {"feature": name}
        for hz in horizons:
            fwd = cl.shift(-hz)/cl - 1; ics = []
            for d in idx:
                a, b = F.loc[d], fwd.loc[d]
                m = a.notna() & b.notna() & (a != 0)
                if m.sum() >= 15:
                    ic = pd.Series(a[m]).corr(pd.Series(b[m]), method="spearman")
                    if not np.isnan(ic): ics.append(ic)
            r[f"IC_{hz}d"] = np.mean(ics) if ics else np.nan
        rows.append(r)
    return pd.DataFrame(rows).set_index("feature")

feats = {"$vol surge (base)": zs(dv_s.rolling(5).mean()/dv_s.rolling(60).mean())}
if HAS_SOC: feats["social accel20"] = SOC_AC; feats["social level"] = SOC_Z
if HAS_SI:  feats["short-int change"] = zs(SI_CHG)
if HAS_OPT: feats["OTM call level"] = OPT
feats["past-meme count"] = PASTM
tab = ic_table(feats)
print("IC (Spearman TB theo ngay). |IC| > 0.02 la co tin hieu that.")
print("Dau AM cung la tin hieu - dung de TRANH thay vi de CHON.\n")
print(tab.round(4).to_string())
fig, ax = plt.subplots(figsize=(9,3.6))
tab["IC_20d"].sort_values().plot(kind="barh", ax=ax,
    color=["#c0392b" if v<0 else "#2e9e5b" for v in tab["IC_20d"].sort_values()])
ax.axvline(0, c="k", lw=.8); ax.axvline(.02, c="gray", ls=":"); ax.axvline(-.02, c="gray", ls=":")
ax.set_title("IC 20 ngay (vach xam = nguong co tin hieu)"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 4 - Backtest tung phien ban
SC = {"base": BASE,
      "si":   BASE + 1.5*zs(SI_CHG).fillna(0) + 1.0*zs(SI_DTC).fillna(0),
      "soc":  BASE + 1.5*SOC_AC - 1.0*SOC_Z,          # level vao voi dau AM (IC am)
      "opt":  BASE + 1.5*OPT,
      "pastm":BASE + 1.5*PASTM,
      "all":  BASE + 1.0*zs(SI_CHG).fillna(0) + 1.5*SOC_AC - 1.0*SOC_Z + 1.0*OPT + 1.5*PASTM}
RANK = {k: v.where(LIQ).rank(axis=1, ascending=False) for k, v in SC.items()}

def _atr(h,l,c,w=14):
    n=len(c); t=np.zeros(n)
    for i in range(1,n): t[i]=max(h[i]-l[i],abs(h[i]-c[i-1]),abs(l[i]-c[i-1]))
    a=np.full(n,np.nan)
    for i in range(w,n): a[i]=t[i-w+1:i+1].mean()
    return a

def backtest(member, window=None):
    """window: (lo, hi) gioi han khoang thoi gian, de so tren cung do phu song."""
    lo_i = 21 if window is None else max(21, DATES.index(window[0]))
    hi_i = len(DATES)-2 if window is None else DATES.index(window[1])
    tr=[]
    for sym in TICKS:
        c=cl[sym].values; v=vo[sym].values; h=hi[sym].values; l=lo[sym].values; o=op[sym].values
        if (~np.isnan(c)).sum() < 200: continue
        atr=_atr(np.nan_to_num(h),np.nan_to_num(l),np.nan_to_num(c)); i=lo_i
        while i < hi_i:
            if np.isnan(c[i]) or np.isnan(o[i+1]): i+=1; continue
            b=np.nanmean(v[i-20:i]); rv=v[i]/b if b>0 else 0
            ma=np.nanmean(c[i-20:i]); ext=c[i]/ma-1 if ma>0 else 0
            typ=(h[i]+l[i]+c[i])/3
            if (rv>=2. and c[i]>typ and c[i]>c[i-1] and ext<=.5 and c[i]>ma
                    and c[i]/c[i-5]-1>.05 and member(DATES[i], sym)):
                ei=i+1; e=o[i+1]
                if e<=0: i+=1; continue
                peak=e; xi=min(ei+60,hi_i); ex=None
                for j in range(ei+1, xi+1):
                    if np.isnan(c[j]): continue
                    sp=e*.92
                    if o[j]<=sp: xi,ex=j,o[j]; break
                    if l[j]<=sp: xi,ex=j,sp; break
                    if (j-ei)>=2 and (np.nanmax(c[ei+1:j+1])/e-1)<.03: xi,ex=j,c[j]; break
                    peak=max(peak,c[j])
                    if c[j]<=peak-3*(atr[j] if not np.isnan(atr[j]) else 0): xi,ex=j,c[j]; break
                    if c[j]>e and c[j]<np.nanmean(c[max(0,j-9):j+1]): xi,ex=j,c[j]; break
                if ex is None or np.isnan(ex): ex=c[xi]
                tr.append({"ticker":sym,"ei":ei,"xi":xi,"entry_date":DATES[ei],
                           "exit_date":DATES[xi],"entry_px":e,"ret":ex/e-1})
                i=xi+1; continue
            i+=1
    td=pd.DataFrame(tr)
    if len(td)==0: return None, None
    td=td.sort_values("entry_date").reset_index(drop=True)
    rows,opens=[],[]
    for _,t in td.iterrows():
        opens=[d for d in opens if d>t["entry_date"]]
        if len(opens)>=MAXPOS: continue
        opens.append(t["exit_date"]); rows.append({**t.to_dict(),"pnl":ALLOC*t["ret"]})
    tk=pd.DataFrame(rows); acc=defaultdict(float)
    for _,t in tk.iterrows():
        c=cl[t["ticker"]].values; sh=ALLOC/t["entry_px"]
        for j in range(int(t["ei"])+1, int(t["xi"])+1):
            if not np.isnan(c[j]) and not np.isnan(c[j-1]): acc[DATES[j]] += sh*(c[j]-c[j-1])
    eq=CAP+pd.Series([acc.get(d,0.) for d in DATES], index=DATES).cumsum()
    r=eq.pct_change().dropna(); yrs=(DATES[-1]-DATES[0]).days/365.25
    dn=r[r<0]
    return dict(n=len(tk), win=(tk.ret>0).mean(), exp=tk.ret.mean(),
                cagr=(eq.iloc[-1]/CAP)**(1/yrs)-1,
                sharpe=r.mean()/r.std()*np.sqrt(252) if r.std()>0 else np.nan,
                sortino=r.mean()/dn.std()*np.sqrt(252) if len(dn) and dn.std()>0 else np.nan,
                dd=((eq-eq.cummax())/eq.cummax()).min(), final=eq.iloc[-1]), eq

def topk(key,k=30):
    R=RANK[key]
    return lambda d,s: (not np.isnan(R.at[d,s])) and R.at[d,s]<=k
rng=np.random.default_rng(1); RS={d:set(rng.choice(TICKS,30,replace=False)) for d in DATES}

TESTS=[("1. HINDSIGHT seed (BIAS)", lambda d,s: s in memeset),
       ("2. PIT gia/volume",        topk("base"))]
if HAS_SI:  TESTS.append(("3. PIT + short interest", topk("si")))
if HAS_SOC: TESTS.append(("4. PIT + social",         topk("soc")))
if HAS_OPT: TESTS.append(("5. PIT + options flow",   topk("opt")))
TESTS.append(("6. PIT + past-meme (hop le)", topk("pastm")))
if HAS_SOC and HAS_OPT: TESTS.append(("7. PIT + tat ca", topk("all")))
TESTS.append(("8. Ngau nhien (control)", lambda d,s: s in RS[d]))

res, curves = {}, {}
for lbl, fn in TESTS:
    m, eq = backtest(fn)
    if m: res[lbl]=m; curves[lbl]=eq
out=pd.DataFrame(res).T[["n","win","exp","cagr","sharpe","sortino","dd","final"]]
print(out.to_string(formatters={"win":"{:.1%}".format,"exp":"{:+.2%}".format,"cagr":"{:.1%}".format,
      "sharpe":"{:.2f}".format,"sortino":"{:.2f}".format,"dd":"{:.1%}".format,"final":"${:,.0f}".format}))
print("\n!! Neu 'Ngau nhien (control)' khong phai thap nhat -> co mau qua nho, chenh lech la NHIEU.")
print("   Luc do tin Cell 3 (IC) hon Cell nay.")

In [ ]:
# Cell 5 - Equity curve tung phien ban
if curves:
    fig, ax = plt.subplots(figsize=(13,5))
    for lbl, eq in curves.items():
        style = "--" if "BIAS" in lbl or "control" in lbl else "-"
        w = 1.0 if style=="--" else 1.8
        ax.plot(eq.index, eq.values, style, lw=w, label=f"{lbl}  (${eq.iloc[-1]:,.0f})")
    ax.axhline(CAP, c="gray", lw=.8)
    ax.set_yscale("log"); ax.set_title("Equity theo tung cach chon universe (log scale)")
    ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print("Net dut = 2 moc doi chieu: HINDSIGHT (tran ao) va NGAU NHIEN (san).")
    print("Phien ban that su tot phai nam TREN duong ngau nhien mot cach on dinh.")